# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.ready_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name'][:20]:20s} {m['vendor'][:11]:12s} \
{m['api'][:11]:12s} {m['model_id'][:12]:14s} {m['status'][:12]:12s}")

|Models available: 57|

  GPT-3.5-Turbo        openai       openai       gpt-3.5-turb   ok          
  Moonshot-v1-8k       moonshot     moonshot     moonshot-v1-   unverified  
  Moonshot-v1-128k     moonshot     moonshot     moonshot-v1-   unverified  
  GPT-4o               openai       openai       gpt-4o-2024-   ok          
  Qwen-Turbo           qwen         qwen         qwen-turbo     unverified  
  o4-mini              openai       openai       o4-mini-2025   temp-omitted
  GPT-4.1              openai       openai       gpt-4.1-2025   ok          
  GPT-4.1-mini         openai       openai       gpt-4.1-mini   ok          
  GPT-4.1-nano         openai       openai       gpt-4.1-nano   ok          
  Kimi-K2              moonshot     moonshot     moonshot-v1-   NAME-ID-MISM
  Qwen3-235B-Instruct  qwen         qwen         qwen3-235b-a   unverified  
  GPT-5                openai       openai       gpt-5-2025-0   temp-omitted
  GPT-5-mini           openai       openai       gpt

### Probed tasks

In [3]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 4|

  1. AUT     :  6894
  2. CWT     :  6715
  3. DAT     :  6452


### Tests availabe

In [4]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Trial-run

In [9]:
models_to_try = ['Claude Haiku 4.5',
'Claude Opus 4.5',
 'Claude Opus 4.7',
 'Claude Opus 5',
 'Claude Sonnet 4.5',
#  'DeepSeek-3.2',
 'DeepSeek-Chat',
 'GPT-3.5-Turbo',
 'GPT-4-Turbo',
 'GPT-4o',
 'GPT-4o-mini',
 'GPT-5.4',
 'Grok 4.2',
 'Grok 4.3',
 'Grok 4.5',
 'Grok Build 0.1',
 'Llama-3.1 8b',
 'Llama-3.2 3b',
 'Llama-4 Maverick',
 'Llama-4 Scout']
models_to_try = [m for m in models if m["name"] in models_to_try]
sorted([m['name'] for m in models_to_try])

['Claude Haiku 4.5',
 'Claude Opus 4.5',
 'Claude Opus 4.7',
 'Claude Opus 5',
 'Claude Sonnet 4.5',
 'GPT-3.5-Turbo',
 'GPT-4-Turbo',
 'GPT-4o',
 'GPT-4o-mini',
 'GPT-5.4',
 'Grok 4.2',
 'Grok 4.3',
 'Grok 4.5',
 'Grok Build 0.1',
 'Llama-3.1 8b',
 'Llama-3.2 3b',
 'Llama-4 Maverick',
 'Llama-4 Scout']

In [ ]:
alp.collect("AUT", models=models_to_try, n_per_model=350,cue='brick')

  GPT-3.5-Turbo: 300 collected, 50 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 50/50 [01:26<00:00,  1.73s/it]


  GPT-4o: 300 collected, 50 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 50/50 [03:24<00:00,  4.09s/it]


  Claude Sonnet 4.5: 300 collected, 50 to collect


AUT:  18%|███████████▌                                                    | 9/50 [01:26<06:24,  9.39s/it]

### Load functions

In [ ]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    text = str(raw or "").strip().strip('"').strip("'")
    tokens = re.split(r"[,\n\r]+", text)
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses = []
    for line in re.split(r"[\n\r]+", str(raw or "")):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_cwt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def load_task(task: str) -> pd.DataFrame:
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "cwt":
        cues = df["prompt"].str.extract(
            r"words: (.+?)\.", expand=False).str.strip().str.split(r",\s*")
        df[["cue_0", "cue_1", "cue_2"]] = pd.DataFrame(cues.tolist()).iloc[:, :3]
        df["response_clean"] = df["raw"].map(parse_cwt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(f"Unknown task: {task}")
    cols = ["task", "model_name", "model_id", 
            "provider", "rep", "temperature_std"] + extra + [
        "prompt", "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)
        
print('All functions loaded...')

### Parse & merge data

In [ ]:
for task in (
#     "dat", 
    "aut",
#     "cwt"
):
    print(f"Parsing {task.upper()}...")
    df = load_task(task)
    print(df.shape)
    df.to_csv(f"./data/{task}.csv", index=False)